# 04_extract — Actor extraction + concept tagging

> **Environment:** requires the project venv **`.venv311`** (Python 3.11) as the Jupyter kernel — the pipeline dependencies are installed only there. `run_all.command` uses it automatically. See `README.md` → Environment setup.

**Input:** `data/interim/sentences_coref.jsonl` (from `03b_coref`)  
**Output:** `data/interim/sentences_tagged.jsonl`

For every sentence produced by `03_preprocess` and coref-resolved by `03b_coref`, this notebook:
1. Re-runs NER on the coref-resolved text, then resolves the surface forms to canonical actor IDs via `src/alias_map.py`, filtered by `ACTOR_WHITELIST`.
2. Tags the sentence with concept clusters via hybrid matching against `src/concept_dict.py`: single-word entries are compared against token **lemmas**, multi-word/hyphenated entries are case-insensitive substring matches on the raw text.
3. Logs every alias resolution and every concept trigger (with its match type) so each tag is traceable to a specific surface form / dictionary term.

## Pipeline steps in this notebook

1. Setup & paths
2. Load sentences + import alias map / concept dictionary
3. Actor extraction function
4. Concept tagging function
4b. Refresh NER on coref-resolved text (from 03b)
5. Apply both functions to all sentences
6. Quality report (top actors, concept rates, alias misses)
7. Spot-check random tagged sentences per concept cluster
8. Write sentences_tagged.jsonl

## Step 1: Setup & paths

In [ ]:
import json
import sys
import random
from pathlib import Path
from collections import Counter, defaultdict

_cwd = Path().resolve()
ROOT = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / 'src').is_dir()),
    _cwd,
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

INTERIM_DIR = ROOT / 'data' / 'interim'
IN_FILE     = INTERIM_DIR / 'sentences_coref.jsonl'
OUT_FILE    = INTERIM_DIR / 'sentences_tagged.jsonl'

print(f'Input  : {IN_FILE}')
print(f'Output : {OUT_FILE}')
assert (ROOT / 'src').is_dir(), f'ERROR: src/ not found under {ROOT}'
assert IN_FILE.exists(),        f'ERROR: {IN_FILE} not found — run 03b_coref first'

## Step 2: Load sentences and shared modules

`ALIAS_MAP`, `ACTOR_WHITELIST` and `CONCEPT_DICT` are our sources.

In [ ]:
from src.alias_map   import ALIAS_MAP, ACTOR_WHITELIST
from src.concept_dict import CONCEPT_DICT

with open(IN_FILE, encoding='utf-8') as f:
    sentences = [json.loads(line) for line in f]

print(f'Loaded {len(sentences)} sentences')

n_missing_lemmas = sum(1 for s in sentences if 'lemmas' not in s)
assert n_missing_lemmas == 0, (
    f'{n_missing_lemmas} sentences lack the "lemmas" field — '
    're-run 03_preprocess (lemma-based concept matching requires it)'
)
print(f'ALIAS_MAP entries       : {len(ALIAS_MAP)}')
print(f'ACTOR_WHITELIST entries : {len(ACTOR_WHITELIST)}')
print(f'CONCEPT_DICT clusters   : {len(CONCEPT_DICT)}')
for cluster, terms in CONCEPT_DICT.items():
    print(f'  {cluster:18s}  {len(terms)} terms')

## Step 3: Actor extraction

Per README §6, the resolution rule is:
1. Lowercase the surface form and look it up in `ALIAS_MAP`.
2. If no alias hit, take the surface upper-snake (e.g. `Khamenei` -> `KHAMENEI`).
3. Discard `EVENT_ANCHOR` resolutions (event names are not actors).
4. Keep only canonicals that appear in `ACTOR_WHITELIST`.

We additionally collect a list of **alias misses** — surface forms that NER tagged with a whitelisted entity label (GPE/ORG/PERSON/NORP) but failed to resolve to a known actor. These are candidates to add to `alias_map.py` or `ACTOR_WHITELIST` after manual inspection.

In [ ]:
def extract_actors(ents):
    """Return (resolved_actors, alias_log, miss_surfaces) for one sentence's ents."""
    resolved  = []
    alias_log = []
    misses    = []
    for surface, label in ents:
        canonical = ALIAS_MAP.get(surface.lower().strip())
        if canonical is None:
            canonical = surface.upper().replace(' ', '_')
        if canonical == 'EVENT_ANCHOR':
            continue  # event-anchor surfaces are intentionally discarded
        if canonical in ACTOR_WHITELIST:
            resolved.append(canonical)
            alias_log.append({'surface': surface, 'resolved': canonical})
        else:
            misses.append(surface)
    return list(set(resolved)), alias_log, misses

## Step 4: Concept tagging

Hybrid matching against `CONCEPT_DICT` (see the header comment in `src/concept_dict.py`):

- **Lemma match** — entries without space/hyphen are compared for equality against the sentence's `lemmas` list, so one entry covers its whole inflection family ("strike" matches strikes/striking/struck; "negotiate" matches negotiated/negotiating).
- **Raw match** — entries containing a space or hyphen ("bunker buster", "cease-fire") remain case-insensitive substring matches on the raw text, exactly as before.

One match per cluster is sufficient — the loop breaks on the first hit so each cluster is logged at most once per sentence, recording the trigger term and its `match_type` (`lemma` or `raw`). A single sentence may match multiple clusters.

In [ ]:
def tag_concepts(text, lemmas):
    text_lower = text.lower()
    lemma_set  = set(lemmas)
    matched     = []
    concept_log = []
    for concept, terms in CONCEPT_DICT.items():
        for term in terms:
            if ' ' in term or '-' in term:
                # multi-word / hyphenated entry -> substring on raw text
                if term.lower() in text_lower:
                    matched.append(concept)
                    concept_log.append({'concept': concept, 'trigger': term, 'match_type': 'raw'})
                    break  # one match per cluster is sufficient
            else:
                # single-word entry -> equality against token lemmas
                if term in lemma_set:
                    matched.append(concept)
                    concept_log.append({'concept': concept, 'trigger': term, 'match_type': 'lemma'})
                    break  # one match per cluster is sufficient
    return list(set(matched)), concept_log

## Step 4b: Refresh NER on coref-resolved text

`03b_coref` rewrote pronouns that point to whitelisted actors into `coref_resolved_text`. Here we re-run spaCy NER on that resolved text so those now-explicit actor mentions are detected and feed actor extraction below.

Only sentences that coref actually changed are re-processed; for the rest `coref_resolved_text == text`, so the entities computed in `03_preprocess` already equal "NER on text" and are reused unchanged (and we fall back to `text` for any record lacking the field).

In [ ]:
import spacy

KEEP_LABELS = {'GPE', 'ORG', 'PERSON', 'NORP'}

# Re-NER only the sentences coref rewrote; reuse 03's ents for the rest (where
# coref_resolved_text == text, 03's ents are exactly "NER on text").
to_ner = [s for s in sentences
          if s.get('coref_resolved_text', s['text']) != s['text']]
print(f'Re-running NER on {len(to_ner)} coref-changed sentence(s) of {len(sentences)}')

if to_ner:
    nlp = spacy.load('en_core_web_trf', disable=['lemmatizer'])
    texts = [s['coref_resolved_text'] for s in to_ner]
    for s, doc in zip(to_ner, nlp.pipe(texts, batch_size=32)):
        s['ents'] = [[e.text, e.label_] for e in doc.ents if e.label_ in KEEP_LABELS]
    print('NER refreshed on coref-resolved sentences (ents updated).')
else:
    print('No coref rewrites to re-NER — using 03 entities unchanged.')

## Step 5: Apply to all sentences

In [ ]:
# Defensive filter: skip empty sentences (spaCy paragraph-break artifacts).
# 03_preprocess now drops these at write time, but older sentences.jsonl
# files still contain them.
before = len(sentences)
sentences = [s for s in sentences if s['text'].strip()]
print(f'Filtered {before - len(sentences)} empty sentences')

tagged = []
actor_counter   = Counter()
concept_counter = Counter()
miss_counter    = Counter()
n_with_actor   = 0
n_with_concept = 0
n_with_both    = 0

for s in sentences:
    actors, alias_log, misses     = extract_actors(s['ents'])
    concepts, concept_log         = tag_concepts(s['text'], s['lemmas'])

    for a in actors:    actor_counter[a]   += 1
    for c in concepts:  concept_counter[c] += 1
    for m in misses:    miss_counter[m]    += 1
    if actors:                  n_with_actor   += 1
    if concepts:                n_with_concept += 1
    if actors and concepts:     n_with_both    += 1

    tagged.append({
        'sentence_id': s['sentence_id'],
        'article_id':  s['article_id'],
        'date':        s['date'],
        'source':      s['source'],
        'window':      s['window'],
        'text':        s['text'],
        'actors':      actors,
        'concepts':    concepts,
        'alias_log':   alias_log,
        'concept_log': concept_log,
    })

print(f'Tagged {len(tagged)} sentences')
print(f'  with at least one actor   : {n_with_actor}   ({100*n_with_actor/max(len(tagged),1):.1f}%)')
print(f'  with at least one concept : {n_with_concept} ({100*n_with_concept/max(len(tagged),1):.1f}%)')
print(f'  with BOTH (edge candidates): {n_with_both}    ({100*n_with_both/max(len(tagged),1):.1f}%)')

## Step 6: Quality report

In [ ]:
print('=== Top actors by total resolved-mention count ===')
for actor, cnt in actor_counter.most_common(20):
    print(f'  {cnt:5d}  {actor}')

missing_actors = sorted(set(ACTOR_WHITELIST) - set(actor_counter.keys()))
if missing_actors:
    print(f'\nWhitelisted actors with ZERO mentions in this corpus: {len(missing_actors)}')
    print('  ' + ', '.join(missing_actors))
    print('  -> expected for actors that only appear in later windows')

print('\n=== Concept match counts ===')
for cluster in CONCEPT_DICT.keys():
    cnt  = concept_counter.get(cluster, 0)
    rate = 100 * cnt / max(len(tagged), 1)
    print(f'  {cluster:18s}  {cnt:5d} sentences  ({rate:.1f}% of corpus)')

print('\n=== Top alias misses (whitelist-eligible NER surfaces that did NOT resolve) ===')
print('Inspect these — promote relevant ones into ALIAS_MAP or ACTOR_WHITELIST in src/alias_map.py')
for surface, cnt in miss_counter.most_common(30):
    print(f'  {cnt:4d}  {surface}')

match_type_counter = Counter(
    e['match_type'] for t in tagged for e in t['concept_log']
)
print('\n=== Concept trigger match types ===')
for mtype, cnt in match_type_counter.most_common():
    print(f'  {mtype:6s}  {cnt}')

## Step 7: Spot-check tagged sentences per concept cluster

Per README §6 validation note: manually inspect 20–30 random tagged sentences per cluster and confirm the trigger term is doing the right work semantically. 
If precision looks poor for a cluster, narrow its term list in `src/concept_dict.py`.

In [ ]:
SAMPLES_PER_CLUSTER = 5  # bump to 20–30 when doing the formal validation pass
random.seed(42)

by_cluster = defaultdict(list)
for t in tagged:
    for c in t['concepts']:
        by_cluster[c].append(t)

for cluster in CONCEPT_DICT.keys():
    pool = by_cluster[cluster]
    print(f'\n========== {cluster}  ({len(pool)} matches) ==========')
    if not pool:
        print('  (no matches in this corpus)')
        continue
    sample = random.sample(pool, min(SAMPLES_PER_CLUSTER, len(pool)))
    for t in sample:
        triggers = [e['trigger'] for e in t['concept_log'] if e['concept'] == cluster]
        print(f"  [{t['sentence_id']}] triggers={triggers}")
        print(f'    actors  : {t["actors"]}')
        print(f'    text    : {t["text"][:240]}')

## Step 8: Write sentences_tagged.jsonl

In [ ]:
with open(OUT_FILE, 'w', encoding='utf-8') as f:
    for t in tagged:
        f.write(json.dumps(t, ensure_ascii=False) + '\n')

print(f'Wrote {len(tagged)} tagged sentences to {OUT_FILE}')
print()
print('VALIDATION CHECKPOINT (04_extract):')
print(f'  Sentences with actor + concept (edge candidates): {n_with_both}')
print(f'  Coverage rate : {100*n_with_both/max(len(tagged),1):.1f}%  (target: 5–20% on news text)')
print()
print('Inspect Step 7 output and the alias-miss list (Step 6) before moving to 05_edges.')
print('Iterate on src/alias_map.py and src/concept_dict.py, then re-run 04 — fast, no spaCy needed.')